%md
## Utilities Dataset Generator

Generates a synthetic **utilities** dataset in Unity Catalog with **realistic statistical distributions** and Faker-generated PII.

Each customer is assigned **1–4 utility services** (Electricity, Natural Gas, Water, Solar), producing one meter per customer-service pair.

### Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `catalog` | `industry_sample_data` | Target Unity Catalog |

### Tables

| Schema | Entity Table | Rows | Event Table | Rows | Key Features |
| --- | --- | --- | --- | --- | --- |
| `utilities` | `meters` | ~5K | `usage_records` | 100K-500K | Multi-utility customers, temperature-driven usage, grid zones, spatially correlated outages |

### How to Run

1. Set the `catalog` widget parameter at the top of the notebook.
2. **Run All** — the notebook creates the `utilities` schema and generates both tables.
3. Final cells apply column comments and a `RemoveAfter` tag for workspace retention compliance.


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

DROP SCHEMA IF EXISTS IDENTIFIER(:catalog || '.utilities') CASCADE;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.utilities');

In [0]:
%pip install faker --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import random
import math
from datetime import datetime, timedelta, date
from faker import Faker
from pyspark.sql import Row

fake = Faker()
Faker.seed(63)
random.seed(63)
CATALOG = dbutils.widgets.get('catalog')
SCHEMA = "utilities"
CATALOG_SCHEMA = f"{CATALOG}.{SCHEMA}"
NOW = datetime.now()

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

# --- Configuration ---
service_types = ["Electricity", "Natural Gas", "Water", "Solar"]
meter_statuses = ["Active", "Inactive", "Faulty", "Pending Replacement"]
regions = ["Northeast", "Southeast", "Midwest", "Southwest", "West Coast", "Pacific Northwest"]
cities = {
    "Northeast": ["New York", "Boston", "Philadelphia", "Hartford", "Providence"],
    "Southeast": ["Miami", "Atlanta", "Charlotte", "Tampa", "Nashville"],
    "Midwest": ["Chicago", "Detroit", "Minneapolis", "Columbus", "Milwaukee"],
    "Southwest": ["Dallas", "Phoenix", "Houston", "San Antonio", "Tucson"],
    "West Coast": ["Los Angeles", "San Francisco", "San Diego", "Sacramento"],
    "Pacific Northwest": ["Portland", "Seattle", "Boise", "Eugene"]}

# Grid zones per region for spatial analysis
region_grid_zones = {r: [f"{r[:2].upper()}-GRID-{z:02d}" for z in range(1, 8)] for r in regions}

service_rate_plans = {
    "Electricity": {"plans": ["Residential Basic", "Residential Time-of-Use", "Commercial Standard", "Commercial Peak", "Industrial"],
                    "weights": [35, 25, 20, 12, 8]},
    "Natural Gas": {"plans": ["Residential Basic", "Commercial Standard", "Commercial Peak", "Industrial"],
                    "weights": [45, 25, 18, 12]},
    "Water":       {"plans": ["Residential Basic", "Residential Time-of-Use", "Commercial Standard"],
                    "weights": [55, 20, 25]},
    "Solar":       {"plans": ["Residential Basic", "Residential Time-of-Use"],
                    "weights": [40, 60]},
}

# --- Generate customers, each with 1-4 utility service types ---
NUM_CUSTOMERS = 2000
customers = []
for c in range(1, NUM_CUSTOMERS + 1):
    region = random.choice(regions)
    city = random.choice(cities[region])
    grid_zone = random.choice(region_grid_zones[region])
    # Each customer subscribes to 1-4 utility services
    num_services = random.choices([1, 2, 3, 4], weights=[15, 35, 35, 15])[0]
    customer_services = random.sample(service_types, num_services)
    customers.append({
        "customer_id": f"CUST-{c:05d}",
        "customer_name": fake.name(),
        "region": region,
        "city": city,
        "grid_zone": grid_zone,
        "services": customer_services,
    })

# --- Meters table (~5K rows, one meter per customer-service pair) ---
meters = []
meter_idx = 0
for cust in customers:
    for svc in cust["services"]:
        meter_idx += 1
        install = date(2012, 1, 1) + timedelta(days=random.randint(0, 5000))
        install_age_years = (NOW.date() - install).days / 365.25
        srp = service_rate_plans[svc]
        rate_plan = random.choices(srp["plans"], weights=srp["weights"])[0]

        if install_age_years < 2: smart_prob = 0.90
        elif install_age_years < 5: smart_prob = 0.70
        elif install_age_years < 8: smart_prob = 0.45
        else: smart_prob = 0.20
        is_smart = random.random() < smart_prob

        if install_age_years > 8:
            m_status = random.choices(meter_statuses, weights=[60, 12, 18, 10])[0]
        elif install_age_years > 5:
            m_status = random.choices(meter_statuses, weights=[75, 10, 10, 5])[0]
        else:
            m_status = random.choices(meter_statuses, weights=[88, 5, 4, 3])[0]

        if svc in ["Electricity", "Solar"]:
            if "Industrial" in rate_plan: cap = round(clamp(random.lognormvariate(5.5, 0.6), 100.0, 2000.0), 1)
            elif "Commercial" in rate_plan: cap = round(clamp(random.lognormvariate(4.0, 0.7), 20.0, 500.0), 1)
            else: cap = round(clamp(random.lognormvariate(2.7, 0.8), 3.0, 50.0), 1)
        else:
            cap = None

        meters.append(Row(
            meter_id=f"MTR-{10000 + meter_idx}",
            customer_id=cust["customer_id"],
            customer_name=cust["customer_name"],
            service_type=svc,
            rate_plan=rate_plan,
            meter_status=m_status,
            region=cust["region"],
            city=cust["city"],
            grid_zone=cust["grid_zone"],
            install_date=install,
            is_smart_meter=is_smart,
            capacity_kw=cap
        ))

meter_lookup = {m.meter_id: m for m in meters}
meter_ids = [m.meter_id for m in meters]

meters_df = spark.createDataFrame(meters)
meters_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.meters")
print(f"\u2714 Created {CATALOG_SCHEMA}.meters ({meters_df.count()} rows)")

# --- Usage Records table (randomized ~100K-500K rows) ---
# Pre-generate outage events per grid_zone per month for spatial correlation
outage_events = {}
for region in regions:
    for gz in region_grid_zones[region]:
        for m in range(1, 13):
            # Some zone-months have high outage probability (storm events)
            if random.random() < 0.08:
                outage_events[(gz, m)] = clamp(random.gauss(0.6, 0.15), 0.3, 0.9)
            else:
                outage_events[(gz, m)] = 0.0

# Regional temperature baselines (avg \u00b0F per month)
temp_baselines = {
    "Northeast":        [28, 31, 40, 52, 63, 73, 78, 76, 68, 56, 44, 32],
    "Southeast":        [48, 52, 58, 66, 74, 80, 83, 82, 77, 67, 57, 50],
    "Midwest":          [22, 26, 38, 52, 64, 74, 78, 76, 67, 54, 40, 27],
    "Southwest":        [52, 56, 62, 72, 82, 92, 96, 94, 88, 76, 62, 54],
    "West Coast":       [52, 54, 56, 60, 64, 68, 72, 72, 70, 64, 56, 52],
    "Pacific Northwest": [38, 40, 44, 50, 56, 62, 68, 68, 62, 52, 44, 38],
}

records = []
NUM_EVENT_RECORDS = random.randint(100_000, 500_000)
for i in range(1, NUM_EVENT_RECORDS + 1):
    reading_ts = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 730), hours=random.randint(0, 23))
    meter_id = random.choice(meter_ids)
    meter = meter_lookup[meter_id]
    svc = meter.service_type
    month = reading_ts.month

    # Temperature: regional baseline with daily noise
    base_temp = temp_baselines.get(meter.region, [55]*12)[month - 1]
    temperature_f = round(clamp(random.gauss(base_temp, 8), base_temp - 25.0, base_temp + 25.0), 1)

    if svc == "Electricity":
        seasonal = 1.0 + 0.4 * math.sin((month - 3) * math.pi / 6)
        # Temperature-driven adjustment: more usage in extreme temps
        temp_factor = 1.0 + max(0, (temperature_f - 85)) * 0.01 + max(0, (32 - temperature_f)) * 0.008
        usage = round(clamp(random.lognormvariate(3.2, 0.7) * seasonal * temp_factor, 2.0, 300.0), 2)
        unit = "kWh"
        rate = round(clamp(random.lognormvariate(-1.8, 0.4), 0.06, 0.50), 4)
    elif svc == "Natural Gas":
        seasonal = 1.0 + 0.5 * math.sin((month - 9) * math.pi / 6)
        temp_factor = 1.0 + max(0, (50 - temperature_f)) * 0.015
        usage = round(clamp(random.lognormvariate(2.5, 0.8) * seasonal * temp_factor, 0.5, 200.0), 2)
        unit = "therms"
        rate = round(clamp(random.lognormvariate(0.1, 0.4), 0.40, 3.50), 4)
    elif svc == "Water":
        seasonal = 1.0 + 0.3 * math.sin((month - 3) * math.pi / 6)
        usage = round(clamp(random.lognormvariate(4.5, 0.8) * seasonal, 5.0, 1500.0), 2)
        unit = "gallons"
        rate = round(clamp(random.lognormvariate(-5.0, 0.5), 0.002, 0.025), 4)
    else:  # Solar
        seasonal = 1.0 + 0.5 * math.sin((month - 3) * math.pi / 6)
        usage = round(clamp(random.gauss(-20.0, 15.0) * seasonal, -80.0, 30.0), 2)
        unit = "kWh"
        rate = round(clamp(random.lognormvariate(-1.8, 0.4), 0.06, 0.50), 4)

    if "Industrial" in meter.rate_plan:
        usage = round(usage * clamp(random.gauss(8.0, 2.0), 4.0, 15.0), 2)
    elif "Commercial" in meter.rate_plan:
        usage = round(usage * clamp(random.gauss(3.5, 1.0), 1.5, 6.0), 2)

    cost = round(abs(usage) * rate, 2)

    if meter.is_smart_meter:
        reading_type = random.choices(["Scheduled", "On-Demand", "Estimated"], weights=[80, 15, 5])[0]
    else:
        reading_type = random.choices(["Scheduled", "On-Demand", "Estimated"], weights=[20, 15, 65])[0]

    if svc in ["Electricity", "Solar"]:
        peak = round(clamp(random.lognormvariate(2.0, 0.8), 0.5, 100.0), 2)
        if "Industrial" in meter.rate_plan: peak = round(peak * 5.0, 2)
        elif "Commercial" in meter.rate_plan: peak = round(peak * 2.5, 2)
    else:
        peak = None

    # Spatially correlated outages: check grid_zone + month events
    zone_outage_rate = outage_events.get((meter.grid_zone, month), 0.0)
    svc_outage_base = {"Electricity": 0.10, "Natural Gas": 0.03, "Water": 0.05, "Solar": 0.02}
    storm_season = 1.5 if month in [6, 7, 8, 9] else 0.8
    base_outage_prob = svc_outage_base.get(svc, 0.05) * storm_season
    combined_outage_prob = min(0.95, base_outage_prob + zone_outage_rate)
    had_outage = random.random() < combined_outage_prob

    outage_min = int(clamp(random.expovariate(0.02), 5, 720)) if had_outage else 0

    records.append(Row(
        record_id=20000 + i,
        meter_id=meter_id,
        reading_date=reading_ts,
        service_type=svc,
        usage_amount=usage,
        usage_unit=unit,
        rate_per_unit=rate,
        cost=cost,
        reading_type=reading_type,
        peak_demand_kw=peak,
        temperature_f=temperature_f,
        had_outage=had_outage,
        outage_duration_minutes=outage_min
    ))

records_df = spark.createDataFrame(records)
records_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.usage_records")
print(f"\u2714 Created {CATALOG_SCHEMA}.usage_records ({records_df.count()} rows)")

print("\n--- Meters (sample) ---")
display(meters_df.limit(5))
print("\n--- Usage Records (sample) ---")
display(records_df.limit(5))

✔ Created industry_sample_data.utilities.meters (5000 rows)
✔ Created industry_sample_data.utilities.usage_records (439376 rows)

--- Meters (sample) ---


meter_id,customer_id,customer_name,service_type,rate_plan,meter_status,region,city,grid_zone,install_date,is_smart_meter,capacity_kw
MTR-10001,CUST-00001,Edward Blankenship,Water,Commercial Standard,Active,Southwest,San Antonio,SO-GRID-03,2019-09-29,false,null
MTR-10002,CUST-00001,Edward Blankenship,Natural Gas,Residential Basic,Active,Southwest,San Antonio,SO-GRID-03,2016-09-08,false,null
MTR-10003,CUST-00001,Edward Blankenship,Electricity,Residential Time-of-Use,Active,Southwest,San Antonio,SO-GRID-03,2016-03-19,false,30.0
MTR-10004,CUST-00001,Edward Blankenship,Solar,Residential Time-of-Use,Active,Southwest,San Antonio,SO-GRID-03,2024-10-04,true,8.3
MTR-10005,CUST-00002,Mr. Paul Jones,Water,Residential Basic,Active,Pacific Northwest,Portland,PA-GRID-01,2013-01-21,false,null



--- Usage Records (sample) ---


record_id,meter_id,reading_date,service_type,usage_amount,usage_unit,rate_per_unit,cost,reading_type,peak_demand_kw,temperature_f,had_outage,outage_duration_minutes
20001,MTR-11317,2025-02-14T02:00:00.000Z,Water,69.94,gallons,0.002,0.14,Estimated,null,25.1,false,0
20002,MTR-11860,2024-09-10T14:00:00.000Z,Solar,-18.25,kWh,0.1508,2.75,Estimated,20.44,71.6,false,0
20003,MTR-10969,2024-12-19T20:00:00.000Z,Electricity,38.7,kWh,0.2281,8.83,On-Demand,7.15,49.7,false,0
20004,MTR-14334,2024-01-04T02:00:00.000Z,Natural Gas,54.79,therms,1.1561,63.34,On-Demand,null,22.2,false,0
20005,MTR-14538,2024-06-27T01:00:00.000Z,Solar,-41.01,kWh,0.069,2.83,Estimated,2.33,80.0,false,0


In [0]:
CATALOG = dbutils.widgets.get('catalog')

def apply_comments(table_fqn, comments):
    for col, comment in comments.items():
        spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN `{col}` COMMENT '{comment}'")
    print(f"\u2714 {table_fqn} \u2014 {len(comments)} column comments applied")

apply_comments(f"{CATALOG}.utilities.meters", {
    "meter_id":       "Unique meter identifier (MTR-XXXXX format)",
    "customer_id":    "Customer identifier (CUST-XXXXX format). Multiple meters share the same customer_id",
    "customer_name":  "Full name of the account holder (generated via Faker)",
    "service_type":   "Utility service: Electricity, Natural Gas, Water, or Solar",
    "rate_plan":      "Billing rate plan (service-type-specific)",
    "meter_status":   "Meter status: Active, Inactive, Faulty, or Pending Replacement",
    "region":         "Geographic service region",
    "city":           "City within the service region",
    "grid_zone":      "Grid zone identifier for spatial analysis (XX-GRID-NN format). Region-correlated",
    "install_date":   "Date the meter was installed (DateType)",
    "is_smart_meter": "Whether this is a smart meter. Newer installs much more likely to be smart",
    "capacity_kw":    "Meter capacity in kilowatts. NULL for Natural Gas and Water meters",
})

apply_comments(f"{CATALOG}.utilities.usage_records", {
    "record_id":              "Unique identifier for the usage reading",
    "meter_id":               "Foreign key referencing meters.meter_id",
    "reading_date":           "Timestamp of the meter reading (TimestampType)",
    "service_type":           "Utility service type matching the meter",
    "usage_amount":           "Usage quantity. Negative for Solar = net generation. Includes seasonal + temperature variation",
    "usage_unit":             "Unit of measurement: kWh, therms, or gallons",
    "rate_per_unit":          "Billing rate per usage unit in USD",
    "cost":                   "Computed cost: abs(usage_amount) x rate_per_unit",
    "reading_type":           "How the reading was obtained: Scheduled, On-Demand, or Estimated",
    "peak_demand_kw":         "Peak demand in kilowatts. NULL for Gas and Water",
    "temperature_f":          "Ambient temperature in Fahrenheit. Regional monthly baselines with daily noise. Drives electricity and gas usage",
    "had_outage":             "Whether a service outage occurred. Spatially correlated by grid_zone and month",
    "outage_duration_minutes":"Duration of outage in minutes. 0 if no outage",
})

print(f"\n\u2705 All column comments applied for utilities schema")

spark.sql(f"ALTER TABLE {CATALOG}.utilities.meters ALTER COLUMN meter_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.utilities.meters ADD CONSTRAINT pk_meters PRIMARY KEY (meter_id)")
spark.sql(f"ALTER TABLE {CATALOG}.utilities.usage_records ALTER COLUMN record_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.utilities.usage_records ADD CONSTRAINT pk_usage_records PRIMARY KEY (record_id)")
spark.sql(f"ALTER TABLE {CATALOG}.utilities.usage_records ADD CONSTRAINT fk_usage_records_meter_id FOREIGN KEY (meter_id) REFERENCES {CATALOG}.utilities.meters(meter_id)")
print(f"\u2714 PK/FK constraints applied for utilities schema")

✔ industry_sample_data.utilities.meters — 12 column comments applied
✔ industry_sample_data.utilities.usage_records — 13 column comments applied

✅ All column comments applied for utilities schema


In [0]:
%sql
COMMENT ON SCHEMA IDENTIFIER(:catalog || '.utilities') IS
'Utilities sample dataset with realistic statistical distributions and Faker-generated PII. Entity table: `meters` (~5K rows, PK: meter_id). Event table: `usage_records` (100K-500K rows, PK: record_id, FK: meter_id → meters). Key features: Temperature-driven usage, grid zones, spatially correlated outages.';

In [0]:
CATALOG = dbutils.widgets.get('catalog')
remove_after_value = "2026-12-31"

spark.sql(f"ALTER SCHEMA `{CATALOG}`.`utilities` SET TAGS ('RemoveAfter' = '{remove_after_value}')")
print(f"✔ RemoveAfter tag applied to utilities schema ({remove_after_value})")

✔ RemoveAfter tag applied to utilities schema (2026-12-31)
